In [1]:
# ==============================================================================
# NOTEBOOK 06: EXPORT GOLD TELEMETRY SNAPSHOTS FOR LOCAL REACT COPILOT
# ==============================================================================
from notebookutils import mssparkutils

# 1. Top 25 Critical Fraud Alerts (P1)
pdf_p1 = spark.sql("""
    SELECT tx_id, step, type, amount, nameOrig, oldbalanceOrg, newbalanceOrig,
           nameDest, errorBalanceOrig, errorBalanceDest, origBalanceDrainRatio,
           orig_tx_count_1h, ml_fraud_probability, risk_tier
    FROM gold_fact_fraud_scored
    WHERE risk_tier = 'P1 - Critical'
    ORDER BY amount DESC
    LIMIT 25
""").toPandas()

# 2. Treasury MILP Optimization Summary
pdf_treasury = spark.read.table("gold_fact_treasury_optimization").drop("optimized_at").toPandas()

# 3. Latest Quarter Peer & ApexFin Financial Benchmark
pdf_peers = spark.sql("""
    SELECT ticker, segment, fiscal_year, fiscal_quarter, gpv_millions,
           net_revenue_millions, operating_margin_pct, take_rate_bps,
           FEDFUNDS, cpi_yoy_pct, yield_spread_10y2y
    FROM gold_fact_corporate_financials
    WHERE fiscal_year >= 2024
    ORDER BY fiscal_year DESC, fiscal_quarter DESC
""").toPandas()

# Write JSON payloads to OneLake Files/copilot_export/
mssparkutils.fs.mkdirs("Files/copilot_export")
mssparkutils.fs.put("Files/copilot_export/top_p1_fraud_alerts.json", pdf_p1.to_json(orient="records", indent=2), True)
mssparkutils.fs.put("Files/copilot_export/treasury_milp_plan.json", pdf_treasury.to_json(orient="records", indent=2), True)
mssparkutils.fs.put("Files/copilot_export/peer_financials_benchmark.json", pdf_peers.to_json(orient="records", indent=2), True)

print(f"Exported top_p1_fraud_alerts.json ({len(pdf_p1)} rows)")
print(f"Exported treasury_milp_plan.json ({len(pdf_treasury)} rows)")
print(f"Exported peer_financials_benchmark.json ({len(pdf_peers)} rows)")

StatementMeta(, 9257418b-a1af-4438-a722-b537d3cb8b91, 3, Finished, Available, Finished, False)

Exported top_p1_fraud_alerts.json (25 rows)
Exported treasury_milp_plan.json (4 rows)
Exported peer_financials_benchmark.json (63 rows)


In [2]:
import base64
from IPython.display import HTML, display

files = [
    ("top_p1_fraud_alerts.json", pdf_p1.to_json(orient="records", indent=2)),
    ("treasury_milp_plan.json", pdf_treasury.to_json(orient="records", indent=2)),
    ("peer_financials_benchmark.json", pdf_peers.to_json(orient="records", indent=2))
]

links_html = "<h4>Click each link below to save directly to your PC:</h4><ul>"
for fname, content in files:
    b64 = base64.b64encode(content.encode("utf-8")).decode("utf-8")
    links_html += f'<li style="margin:8px 0;"><a download="{fname}" href="data:application/json;base64,{b64}" style="font-size:16px; font-weight:bold; color:#0078D4;">Download {fname}</a></li>'
links_html += "</ul>"

display(HTML(links_html))

StatementMeta(, 9257418b-a1af-4438-a722-b537d3cb8b91, 4, Finished, Available, Finished, False)